In [0]:
source_path = "abfss://raw@financestoragebablu.dfs.core.windows.net/"
bronze_path = "abfss://bronze@financestoragebablu.dfs.core.windows.net/"

folders = [f.name.replace("/", "") for f in dbutils.fs.ls(source_path) if f.isDir()]
print(f"Found folders: {folders}")

streams = []

for folder in folders:

    print(f"Processing: {folder}")

    df = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("header", "true") \
        .option("cloudFiles.schemaLocation", f"{bronze_path}checkpoint_{folder}") \
        .option("cloudFiles.schemaEvolutionMode", "rescue") \
        .load(f"{source_path}{folder}")

    query = df.writeStream.format("parquet") \
        .outputMode("append") \
        .option("checkpointLocation", f"{bronze_path}checkpoint_{folder}") \
        .option("path", f"{bronze_path}{folder}") \
        .trigger(availableNow=True) \
        .start()

    streams.append(query)

for s in streams:
    s.awaitTermination()

print("\n All streams completed")

In [0]:
bronze_path = "abfss://bronze@financestoragebablu.dfs.core.windows.net/stock"

df=spark.read.format('parquet').load(bronze_path)
display(df)

In [0]:
# =========================
# Event Hub Streaming Ingestion (via Kafka protocol)
# =========================

bronze_path = "abfss://bronze@financestoragebablu.dfs.core.windows.net/"

# Ensure catalog and schema exist
spark.sql("CREATE CATALOG IF NOT EXISTS finance")
spark.sql("CREATE SCHEMA IF NOT EXISTS finance.bronze")

# Get connection string from secrets
eh_conn_str = dbutils.secrets.get("kv-scope", "eh-connection-string")
eh_namespace = "stockeventhubbablu"
eh_topic = "stocktopic"

# Kafka SASL config for Event Hub
sasl_jaas = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{eh_conn_str}";'

# Read stream from Event Hub via Kafka
eh_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{eh_namespace}.servicebus.windows.net:9093")
    .option("subscribe", eh_topic)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", sasl_jaas)
    .option("kafka.request.timeout.ms", "60000")
    .option("kafka.session.timeout.ms", "60000")
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

# Cast key and value from binary to string, keep metadata
eh_df = eh_stream.selectExpr(
    "CAST(key AS STRING) as key",
    "CAST(value AS STRING) as value",
    "topic",
    "partition",
    "offset",
    "timestamp"
)

# Write to Delta table in Unity Catalog
query = (
    eh_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{bronze_path}checkpoint_{eh_topic}")
    .trigger(availableNow=True)
    .toTable("finance.bronze.stocktopic")
)

query.awaitTermination()
print(f"\n Event Hub stream written to finance.bronze.stocktopic")

In [0]:
%sql
SELECT * FROM finance.bronze.stocktopic LIMIT 100